# WI-07: interactive model comparison

Approved protocol: U-Net reference, SegFormer-B0, gated DINOv3, and the reviewed fourth-slot fallback. Keep test untouched until a real stakeholder freeze. Run one training process at a time and retain complete epochs on Drive. See `docs/15_MODEL_SELECTION_EVALUATION.md`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, json
DRIVE_ROOT = Path('/content/drive/MyDrive/Graphene')
REPO = Path('/content/Graphene-Segmentation')
# Use the reviewed immutable commit recorded in the WI-07 verification record.
REVIEWED_COMMIT = '33eb57af378b32ed80a552c44119c4ab7e805ff7'
assert len(REVIEWED_COMMIT) == 40


## Install the reviewed source

Choose a T4 GPU runtime. Package installation can take several minutes. Training CLI processes import the pinned environment fresh; do not reuse model objects from Colab’s preinstalled Torch.

In [ ]:
subprocess.run(['git','clone','--branch','feat/wi-07-model-selection','https://github.com/camiloandcu/Graphene-Segmentation.git',str(REPO)],check=True)
subprocess.run(['git','checkout',REVIEWED_COMMIT],cwd=REPO,check=True)
packages=[REPO/'packages'/n for n in ('graphene-model-contract','graphene-dataset-contract','graphene-training','graphene-evaluation')]
subprocess.run([sys.executable,'-m','pip','install',*[str(p) for p in packages]],check=True)
subprocess.run([sys.executable,'-c',"import torch; print(torch.__version__,torch.cuda.is_available()); assert torch.cuda.is_available()"],check=True)


## Restore the verified dataset

Dataset extraction does not authorize test evaluation. Training preflight consumes only approved train/validation roles.

In [ ]:
import hashlib, zipfile
ZIP_SHA='4aa3bc1ad4d29181338901d773d602613fb2bdc6613ea5d18155e28534ebbea7'
archive=DRIVE_ROOT/'datasets'/f'{ZIP_SHA}.zip'
assert hashlib.sha256(archive.read_bytes()).hexdigest()==ZIP_SHA
DATASET_PARENT=Path('/content/wi07-data'); DATASET_PARENT.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(DATASET_PARENT)
DATASET=DATASET_PARENT/'ready-dataset'


## Configure one admitted experiment

Record license/access and initialization evidence first. DINO requires actual approved weight access. Do not substitute random initialization when a gate fails. Initial runs use the common protocol; later changes need the approved shortlist review. Give each seed/configuration a new run ID.

In [ ]:
RUN_ID='wi07-segformer-b0-seed42-REPLACE_DATE'
RUN=Path('/content')/RUN_ID
PERSIST=DRIVE_ROOT/'runs'/RUN_ID
CONFIG=Path('/content')/f'{RUN_ID}.json'
configuration={'schema_version':2,'architecture':'segformer-b0','weights':'nvidia/mit-b0',
 'weights_revision':'80983a413c30d36a39c20203974ae7807835e2b4',
 'license_approved':False, # Set only after recorded research/evaluation license review.
 'input_size':512,'batch_size':2,'epochs':30,'seed':42,'device':'cuda','precision':'float32',
 'persistence_directory':str(PERSIST)}
CONFIG.write_text(json.dumps(configuration,indent=2))


In [ ]:
subprocess.run([sys.executable,'-m','graphene_evaluation.experiments','admit',str(DATASET),
 '--config',str(CONFIG),'--output',str(DRIVE_ROOT/'trials'/f'{RUN_ID}-admission.json')],check=True)


## Run or recover

Use the first cell for a new run. For a later session, restore the reviewed source/environment and copied durable run, then use the recovery cell. Verify checkpoint completion before resuming; incomplete epochs remain ignored. Keep the interactive notebook open. Do not start another process until this one has ended.

In [ ]:
subprocess.run([sys.executable,'-m','graphene_evaluation.experiments','run',str(DATASET),
 '--config',str(CONFIG),'--output',str(RUN)],check=True)


In [ ]:
# Recovery alternative: do not execute after the new-run cell.
import shutil
assert not RUN.exists()
shutil.copytree(PERSIST,RUN)
complete=sorted(p.name for p in (RUN/'epochs').iterdir() if (p/'complete.json').exists())
assert complete
subprocess.run([sys.executable,'-m','graphene_evaluation.experiments','resume',str(DATASET),
 '--run',str(RUN),'--checkpoint',complete[-1]],check=True)


## Development evidence

Use the actual selected generation from the completed run. Score only validation; copy reports and the run ledger to Drive at session boundaries. Inspect the panels with the stakeholder before choosing a shortlist, geometry change or final checkpoint. Test scoring is intentionally absent from this notebook.

In [ ]:
from graphene_training.storage import inspect
status=inspect(RUN)
REPORT=DRIVE_ROOT/'trials'/f'{RUN_ID}-validation'
subprocess.run([sys.executable,'-m','graphene_evaluation.cli','score',str(DATASET),
 '--run',str(RUN),'--checkpoint',status['best'],'--device','cuda','--output',str(REPORT)],check=True)
